In [ ]:
# 1. Clone & install
!git clone https://github.com/YOUR_USERNAME/playlogic.git
%cd playlogic
!pip install -q openenv numpy pydantic matplotlib transformers datasets accelerate peft trl bitsandbytes

# 2. Imports
import json, torch, random, numpy as np
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from playlogic.server.environment import MultiAgentFootball, Action

# 3. Load model (4‑bit)
model_name = "google/gemma-2b-it"
bnb_config = BitsAndBytesConfig(load_in_4bit=True)
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto", token=True)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# 4. Prompt builder
def build_prompt(local_obs, team_name):
    return f"You are a {team_name} player.\n{local_obs}\nYour next action:"

# 5. Generate action for one player
def generate_action(local_obs, team_name):
    prompt = build_prompt(local_obs, team_name)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=10, temperature=0.8, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    action = tokenizer.decode(outputs[0], skip_special_tokens=True).split("Your next action:")[-1].strip()
    return action

# 6. Full episode rollout (self-play)
def rollout_episode(env):
    obs = env.reset()
    total_rewards = {'A':0, 'B':0}
    done = False
    while not done:
        actions = {}
        # Get joint action from LLM for all 22 players
        obs_dict = json.loads(obs.text)
        for pid, local_obs in obs_dict.items():
            team = 'Team A' if pid in [str(i) for i in range(11)] else 'Team B'
            actions[int(pid)] = generate_action(local_obs, team)
        obs, rewards, done, _ = env.step(Action(text=json.dumps(actions)))
        total_rewards['A'] += rewards['A']
        total_rewards['B'] += rewards['B']
    return total_rewards

# 7. GRPO reward function (simplified: uses team A's reward as objective)
def reward_func(prompts, completions):
    """Custom reward: we ignore prompts/completions here and just use the environment directly because GRPO expects a list of rewards per completion.
       We'll run a full episode per 'completion' and return team A's reward.
    """
    env = MultiAgentFootball()
    ep_reward_A = rollout_episode(env)['A']
    return [ep_reward_A] * len(completions)  # same reward for all completions? In GRPO, each completion should have its own reward.
    # For proper training, we need to run the environment for each completion.
    # This is a placeholder; a real implementation would batch multiple completions globally.

# 8. Dummy dataset
dataset = Dataset.from_dict({"prompt": ["dummy"]*10})

# 9. GRPO config
config = GRPOConfig(
    output_dir="./grpo_football",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    logging_steps=1,
    num_generation_per_prompt=2,
)

# 10. Trainer
trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    args=config,
    train_dataset=dataset,
    reward_func=reward_func,
)

# 11. Train
trainer.train()

# 12. Plot rewards
logs = trainer.state.log_history
rewards = [log.get("reward") for log in logs if "reward" in log]
import matplotlib.pyplot as plt
plt.plot(rewards)
plt.title("Team A Reward per Step")
plt.savefig("reward_plot.png")
plt.show()